# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets in the dataset and their field @ids
record_sets = dataset.record_sets
print(f"Record Sets in the Dataset (referenced by @id):\n------------------------")
for rs in record_sets:
    rs_id = rs['@id']
    rs_name = rs.get('name', rs_id)
    print(f"RecordSet: {rs_name} (@id={rs_id})")
    print("Fields:")
    for field in rs.get('field', []):
        field_id = field['@id']
        field_name = field.get('name', field_id)
        print(f"- {field_name} (@id={field_id})")
    print()
# Print a sample record from each record set
for rs in record_sets:
    print(f"Sample record from {rs['@id']}:")
    for rec in dataset.records(record_set=rs['@id']):
        print(rec)
        break  # Only print the first sample

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Entities are referenced strictly by their `@id` fields.

In [ ]:
# Extract data from each record set into pandas DataFrames
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns for {rs_id}:", df.columns.tolist())
    print(df.head(2))
# Pick the first record set for analysis (update as needed)
primary_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter by field, normalize numeric values, group by categorical attribute.
All fields referenced by their `@id`.

In [ ]:
# EDA: Filter, Normalize, Group
# Identify a numeric field and a group field from the record set
df = dataframes.get(primary_record_set_id)
if df is not None:
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    group_fields = [col for col in df.columns if df[col].dtype == 'object']
    # Example: Use GAD-7 score (if present); Otherwise, pick first numeric field
    numeric_field_id = next((col for col in numeric_fields if 'gad_7_score' in col.lower()), numeric_fields[0]) if numeric_fields else None
    group_field_id = next((col for col in group_fields if 'village' in col.lower()), group_fields[0]) if group_fields else None
    if numeric_field_id:
        threshold = 10  # Change as appropriate for the field
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

        # Group by categorical field
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print('No DataFrame loaded for EDA.')

## 5. Visualization
Visualize the distribution of a numeric field and the grouping by a categorical field using matplotlib.
Fields are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
This notebook illustrated how to load and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using `mlcroissant`.
- The dataset provides rich survey information with demographic and mental health indicator fields referenced by their `@id`.
- We filtered and normalized values for mental health scores, and visualized group differences across villages or similar attributes.
- These techniques can be extended for further statistical modeling, fairness assessment, and community insight generation.